# Packet capture and traffic sniffing  (MODEL SOLUTION, instructor copy, do not distribute)

# Coding problem, 3.1.d: Packet capture credential-leak detector

**Skill:** Detect Attacks · **Maps to:** 3.1.B, 3.1.C · **Language:** Python 3 (pytest)

## Problem
You are triaging a captured packet log at Crestfield Analytics. For each entry, determine whether
it exposes a plaintext credential ("LEAK") or not ("SAFE"):

- **Encrypted ports are always "SAFE"**, regardless of payload content: port `443` (HTTPS) and
  port `22` (SSH) encrypt their payload, so nothing in the payload text can be treated as an
  actual leaked credential.
- **Cleartext-capable ports** are `80` (HTTP), `23` (Telnet), and `21` (FTP). An entry on one of
  these ports is **"LEAK"** if its payload contains any credential indicator string (case-sensitive
  substring match), otherwise it is **"SAFE"**.
- Credential indicator strings to check for (substring match anywhere in the payload):
  `"password="`, `"Password:"`, `"PASS "`.
- Any other port not listed above is treated as `"SAFE"` (out of scope for this detector).

## Function contract
Implement `solve(packets)` in the code editor (see the Reference section below).

## Input
- `packets`: `list[dict]`, each with:
  - `"port"`: `int`, destination port
  - `"payload"`: `str`, the captured payload text

## Output
A `list[str]` of classifications, **one per input packet, in the same order as the input**. Each
entry is one of `"LEAK"`, `"SAFE"`.

## Examples
```python
packets = [
    {"port": 80, "payload": "POST /login HTTP/1.1\r\n\r\nuser=alice&password=Pass123!"},
    {"port": 443, "payload": "\\x16\\x03\\x03 encrypted application data"},
]
solve(packets)  ->  ["LEAK", "SAFE"]

packets2 = [
    {"port": 23, "payload": "login: admin\r\nPassword: R0uterAdmin!"},
    {"port": 80, "payload": "GET /index.html HTTP/1.1"},
]
solve(packets2)  ->  ["LEAK", "SAFE"]
```

## Constraints
- Deterministic, offline, standard library only.
- Port check first: if `port` is `443` or `22`, the result is always `"SAFE"`.
- Otherwise, if `port` is `80`, `23`, or `21`: check for any of `"password="`, `"Password:"`,
  `"PASS "` as a substring of `payload`; if found, `"LEAK"`, else `"SAFE"`.
- Any other port: always `"SAFE"`.
- Must pass all cases in the tests (visible + hidden).

## Reference
Any implementation that satisfies this interface can be swapped in (reference, student, or
composite) and must pass the tests unchanged.

### Entry point
```python
def solve(packets: list[dict]) -> list[str]
```

### Input (`packets`)
`list[dict]` where each dict has:
- `"port"`: `int`, destination port
- `"payload"`: `str`, captured payload text

### Output (`result`)
A `list[str]` the same length as `packets`, in the **same order as the input**, where each element
is one of `"LEAK"`, `"SAFE"`.

### Classification rules
1. If `port` is `443` or `22`, result is `"SAFE"` (encrypted; payload content is never checked).
2. Else if `port` is `80`, `23`, or `21`: result is `"LEAK"` if `payload` contains any of the
   substrings `"password="`, `"Password:"`, `"PASS "`; otherwise `"SAFE"`.
3. Else (any other port), result is `"SAFE"`.

### Guarantees / rules
- Pure function, deterministic, no I/O, no network. Standard library only.
- Output order matches input order exactly.
- Substring checks are case-sensitive exact matches of the three literal strings above.

### Acceptance
Your code passes when every test below shows a checkmark (some are hidden until you run them).


In [ ]:
# Reference solution
"""Reference solution for 3.1.d. MUST pass 100% of the tests.
Also the default 'blackbox' implementation students can swap out (see the hidden grading tests)."""
from typing import List, Dict

_ENCRYPTED_PORTS = {443, 22}
_CLEARTEXT_PORTS = {80, 23, 21}
_INDICATORS = ("password=", "Password:", "PASS ")


def solve(packets: List[Dict]) -> List[str]:
    """Classify each packet as LEAK (cleartext credential exposed) or SAFE.

    Rules (see the Reference section below):
      1. port in {443, 22} -> always SAFE.
      2. port in {80, 23, 21} -> LEAK if payload contains any credential indicator, else SAFE.
      3. any other port -> SAFE.
    """
    result = []
    for p in packets:
        port = p["port"]
        if port in _ENCRYPTED_PORTS:
            result.append("SAFE")
        elif port in _CLEARTEXT_PORTS:
            payload = p["payload"]
            leaked = any(ind in payload for ind in _INDICATORS)
            result.append("LEAK" if leaked else "SAFE")
        else:
            result.append("SAFE")
    return result


## Self-check tests
Run the cell below to check your `solve` implementation above.

In [ ]:
import __main__ as impl  # the functions defined above

def _p(port, payload):
    return {"port": port, "payload": payload}


# ---- visible cases -------------------------------------------------------

def test_http_login_leak():
    packets = [_p(80, "POST /login HTTP/1.1\r\n\r\nuser=alice&password=Pass123!")]
    assert impl.solve(packets) == ["LEAK"]


def test_https_always_safe():
    packets = [_p(443, "password= this looks like a leak but is encrypted TLS payload")]
    assert impl.solve(packets) == ["SAFE"]


def test_telnet_password_leak():
    packets = [_p(23, "login: admin\r\nPassword: R0uterAdmin!")]
    assert impl.solve(packets) == ["LEAK"]


def test_http_no_credential_safe():
    packets = [_p(80, "GET /index.html HTTP/1.1")]
    assert impl.solve(packets) == ["SAFE"]


def test_ftp_pass_leak():
    packets = [_p(21, "USER backup\r\nPASS Backup#2024")]
    assert impl.solve(packets) == ["LEAK"]


def test_ssh_always_safe():
    packets = [_p(22, "SSH-2.0-OpenSSH_9.3 key exchange")]
    assert impl.solve(packets) == ["SAFE"]


def test_out_of_scope_port_safe():
    packets = [_p(53, "DNS query: mail.example A?")]
    assert impl.solve(packets) == ["SAFE"]


def test_empty_packets():
    assert impl.solve([]) == []


def test_mixed_batch_order_preserved():
    packets = [
        _p(80, "password=leaked1"),
        _p(443, "password=leaked2"),
        _p(21, "PASS ftpsecret"),
    ]
    assert impl.solve(packets) == ["LEAK", "SAFE", "LEAK"]


# ---- hidden / edge cases -------------------------------------------------

def test_case_sensitivity_lowercase_pass_not_matched():
    """Lowercase 'pass ' (not the exact 'PASS ' indicator) should NOT trigger a leak."""
    packets = [_p(21, "this text says pass phrase but not the exact marker")]
    assert impl.solve(packets) == ["SAFE"]


def test_https_with_literal_password_text_still_safe():
    packets = [_p(443, "Password: totally-visible-but-port-is-encrypted")]
    assert impl.solve(packets) == ["SAFE"]


def test_multiple_indicators_still_single_leak():
    packets = [_p(80, "password=abc\r\nPassword: def\r\nPASS ghi")]
    assert impl.solve(packets) == ["LEAK"]


def test_unrelated_cleartext_port_content_safe():
    packets = [_p(23, "just some banner text, no credentials here")]
    assert impl.solve(packets) == ["SAFE"]


def test_large_batch_deterministic():
    packets = []
    expected = []
    for i in range(15):
        if i % 3 == 0:
            packets.append(_p(80, f"password=leak{i}"))
            expected.append("LEAK")
        elif i % 3 == 1:
            packets.append(_p(443, f"password=leak{i}"))
            expected.append("SAFE")
        else:
            packets.append(_p(9999, f"password=leak{i}"))
            expected.append("SAFE")
    assert impl.solve(packets) == expected


# ---- run every test above ------------------------------------------------
_results = []
for _name, _fn in list(globals().items()):
    if _name.startswith("test_") and callable(_fn):
        try:
            _fn()
            _results.append((_name, None))
        except Exception as _e:
            _results.append((_name, _e))
for _name, _e in _results:
    print(("PASS  " if _e is None else "FAIL  ") + _name + ("" if _e is None else f": {_e!r}"))
print(f"{sum(e is None for _, e in _results)}/{len(_results)} tests passed")
